In [4]:
import os
import sys
import json
import re

project_root = "C:\\Users\\User\\pg-screening-system"
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import fitz
import pandas as pd
from src.preprocessing.segmenter import segment_document
from src.extractors.gpa_extractor import extract_gpa
from src.extractors.university_extractor import (
    extract_university, load_qs_db, split_bsc_msc
)
from src.extractors.publication_extractor import (
    extract_publications, load_reference_data
)

# Charger les bases
df_qs, qs_names       = load_qs_db("data/qs_rankings_2025.csv")
df_scopus, df_core    = load_reference_data("data/scopus_sjr.csv",
                                             "data/core_rankings.csv")

# Charger le CV Esra
doc      = fitz.open("data/raw/cvs/257264722 CV.pdf")
text_cv  = "".join(p.get_text() for p in doc)
seg      = segment_document(text_cv)

print("Sections :", seg.detected_section_names)
print("Confiance segmentation :", seg.confidence)

ImportError: cannot import name 'split_bsc_msc' from 'src.extractors.university_extractor' (C:\Users\User\pg-screening-system\src\extractors\university_extractor.py)

In [5]:
# Extraire toutes les métriques NLP
edu_text = seg.sections.get("EDUCATION", "")
pub_text = seg.sections.get("PUBLICATIONS", "")

# GPA
parts    = split_bsc_msc(edu_text)
gpa_bsc  = extract_gpa(parts['bsc'], degree_level="bsc")
gpa_msc  = extract_gpa(parts['msc'], degree_level="msc")

# Universités
univ_bsc = extract_university(parts['bsc'], df_qs, qs_names, "bsc")
univ_msc = extract_university(parts['msc'], df_qs, qs_names, "msc")

# Publications
pubs = extract_publications(pub_text, df_scopus, df_core,
                             candidate_hint="Al-Nashash")

print("=== RÉSULTATS NLP ===")
print(f"BSc GPA    : {gpa_bsc.normalised_gpa} (conf={gpa_bsc.confidence})")
print(f"MSc GPA    : {gpa_msc.normalised_gpa} (conf={gpa_msc.confidence})")
print(f"BSc Univ   : {univ_bsc.detected_name} | QS={univ_bsc.qs_rank} (conf={univ_bsc.confidence})")
print(f"MSc Univ   : {univ_msc.detected_name} | QS={univ_msc.qs_rank} (conf={univ_msc.confidence})")
print(f"Pubs       : {len(pubs)} trouvées")
for p in pubs:
    print(f"  → {p.title[:50]}... | conf={p.confidence}")

NameError: name 'seg' is not defined

In [6]:
from dataclasses import dataclass, field
from typing import Optional, List

CONFIDENCE_THRESHOLD = 0.70   # seuil configurable

@dataclass
class FieldConfidence:
    field_name:  str
    value:       object
    confidence:  float
    needs_llm:   bool
    source:      str          # nlp | llm | manual

@dataclass
class GlobalConfidenceReport:
    fields:          List[FieldConfidence]
    global_score:    float
    fields_ok:       List[str]
    fields_llm:      List[str]
    trigger_llm:     bool

def compute_confidence_report(
        gpa_bsc, gpa_msc,
        univ_bsc, univ_msc,
        pubs,
        threshold: float = CONFIDENCE_THRESHOLD
) -> GlobalConfidenceReport:

    fields = [
        FieldConfidence("bsc_gpa",
                        gpa_bsc.normalised_gpa,
                        gpa_bsc.confidence,
                        gpa_bsc.confidence < threshold,
                        "nlp"),
        FieldConfidence("msc_gpa",
                        gpa_msc.normalised_gpa,
                        gpa_msc.confidence,
                        gpa_msc.confidence < threshold,
                        "nlp"),
        FieldConfidence("bsc_university",
                        univ_bsc.detected_name,
                        univ_bsc.confidence,
                        univ_bsc.confidence < threshold,
                        "nlp"),
        FieldConfidence("msc_university",
                        univ_msc.detected_name,
                        univ_msc.confidence,
                        univ_msc.confidence < threshold,
                        "nlp"),
    ]

    # Ajouter les publications
    for i, pub in enumerate(pubs[:3]):
        fields.append(FieldConfidence(
            f"publication_{i+1}",
            pub.title,
            pub.confidence,
            pub.confidence < threshold,
            "nlp"
        ))

    global_score = round(
        sum(f.confidence for f in fields) / len(fields), 3)

    fields_ok  = [f.field_name for f in fields if not f.needs_llm]
    fields_llm = [f.field_name for f in fields if f.needs_llm]

    return GlobalConfidenceReport(
        fields=fields,
        global_score=global_score,
        fields_ok=fields_ok,
        fields_llm=fields_llm,
        trigger_llm=len(fields_llm) > 0
    )

report = compute_confidence_report(
    gpa_bsc, gpa_msc, univ_bsc, univ_msc, pubs)

print("=== CONFIDENCE REPORT ===")
print(f"Score global : {report.global_score}")
print(f"Seuil        : {CONFIDENCE_THRESHOLD}")
print()
print(f"Champs OK ({len(report.fields_ok)}) :")
for f in report.fields_ok:
    print(f"  ✅ {f}")
print()
print(f"Champs → LLM ({len(report.fields_llm)}) :")
for f in report.fields_llm:
    print(f"  ⚠️  {f}")
print()
print(f"Déclencher LLM : {report.trigger_llm}")

NameError: name 'gpa_bsc' is not defined

In [7]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral"

def call_ollama(section_text: str, fields_needed: List[str]) -> dict:
    """
    Appelle Ollama Mistral pour extraire les champs manquants.
    Retourne un dict JSON avec les valeurs extraites.
    """
    schema = {f: None for f in fields_needed}

    prompt = f"""You are an academic CV parser.
Extract ONLY what is explicitly written in the text below.
Return ONLY valid JSON. Use null for absent fields.

Fields to extract: {fields_needed}

Expected JSON format:
{json.dumps(schema, indent=2)}

Text:
{section_text[:1500]}

Return JSON only:"""

    try:
        response = requests.post(OLLAMA_URL, json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.1}
        }, timeout=30)

        if response.status_code == 200:
            raw = response.json().get("response", "")
            # Extraire le JSON de la réponse
            json_match = re.search(r'\{.*\}', raw, re.DOTALL)
            if json_match:
                return json.loads(json_match.group(0))
        return {}

    except requests.exceptions.ConnectionError:
        print("⚠️  Ollama non disponible — démarrez avec : docker compose up -d ollama")
        return {}
    except Exception as e:
        print(f"⚠️  Erreur LLM : {e}")
        return {}


# Test — appel LLM seulement si trigger_llm = True
if report.trigger_llm:
    print(f"Champs à extraire via LLM : {report.fields_llm}")
    print("Appel Ollama...")

    section_for_llm = seg.sections.get("EDUCATION", "") + "\n" + pub_text
    llm_result = call_ollama(section_for_llm, report.fields_llm)

    print("Réponse LLM :")
    print(json.dumps(llm_result, indent=2))
else:
    print("NLP suffisant — LLM non déclenché")
    llm_result = {}

NameError: name 'report' is not defined

In [8]:
def fuse_results(report: GlobalConfidenceReport,
                 llm_result: dict) -> dict:
    """
    Règles de fusion :
    1. NLP conf ≥ 0.70 → garder NLP
    2. NLP conf < 0.70 ET LLM disponible → prendre LLM
    3. Les deux échouent → NULL + needs_review
    """
    final = {}

    for f in report.fields:
        if not f.needs_llm:
            # NLP suffisant
            final[f.field_name] = {
                "value":  f.value,
                "source": "nlp",
                "confidence": f.confidence
            }
        elif f.field_name in llm_result and llm_result[f.field_name]:
            # LLM a trouvé quelque chose
            final[f.field_name] = {
                "value":  llm_result[f.field_name],
                "source": "llm",
                "confidence": 0.65   # confiance LLM par défaut
            }
        else:
            # Les deux ont échoué
            final[f.field_name] = {
                "value":  None,
                "source": "failed",
                "confidence": 0.0,
                "needs_review": True
            }

    return final

fused = fuse_results(report, llm_result)

print("=== RÉSULTAT FINAL FUSIONNÉ ===")
for field, data in fused.items():
    icon = "✅" if data['source'] == "nlp" else \
           "🤖" if data['source'] == "llm" else "❌"
    print(f"{icon} {field:<20} : {data['value']} "
          f"(source={data['source']}, conf={data['confidence']})")

NameError: name 'report' is not defined

In [9]:
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = r.json().get("models", [])
    print("Ollama disponible ✅")
    print("Modèles :", [m['name'] for m in models])
except:
    print("Ollama non disponible ⚠️")
    print("Lancez : docker compose up -d ollama")
    print("Puis   : docker exec pgars-ollama ollama pull mistral")


Ollama disponible ✅
Modèles : ['mistral:7b-instruct']
